# 05 — Evaluation: Metrics, Summary, and the Full Journey

**By the end of this notebook** you'll have run all four evaluation metrics from `src/pragma_encoder/evaluation/metrics.py`, produced a results table summarising what you built across the five-notebook journey, and seen a single chart that puts it all together.

## What this notebook teaches

- The four evaluation metrics PRAGMA uses: AUC-ROC, PR-AUC, F1, and KS statistic
- When to use each metric and what it measures
- How to call `compute_auc`, `compute_pr_auc`, `compute_f1`, `compute_ks` from `pragma_encoder.evaluation.metrics`
- A consolidated results table covering all five notebooks
- Pointers to the next step: real training on IBM TabFormer data

## Prerequisites

Run notebooks 01–04 first. This notebook is the capstone — it re-creates the synthetic classification task from notebooks 03–04 in one place and applies all four metrics to it.

**How to use:** run every cell in order with Shift+Enter.

In [ ]:
# ── Setup: repo root on sys.path ─────────────────────────────────────────────
import sys, pathlib
repo_root = pathlib.Path().resolve().parent   # notebooks/ → repo root
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch

# ── PRAGMA model + adaptation ─────────────────────────────────────────────────
from pragma_encoder.model           import PRAGMA, PRAGMAConfig
from pragma_encoder.model.assembler import EmbeddingAssembler
from pragma_encoder.masking         import MaskingStrategy
from pragma_encoder.tokenizer.vocabulary import VocabularySpec
from pragma_encoder.adaptation.probe    import EmbeddingProbe

# ── PRAGMA evaluation metrics (src/pragma_encoder/evaluation/metrics.py) ─────
from pragma_encoder.evaluation.metrics import (
    compute_auc,     # AUROC  — probability that positive scores higher than negative
    compute_pr_auc,  # PR-AUC — precision-recall area, important for imbalanced data
    compute_f1,      # F1     — harmonic mean of precision and recall (threshold-based)
    compute_ks,      # KS     — max separation between cumulative distributions
)

matplotlib.rcParams['figure.dpi'] = 110
torch.manual_seed(42)
np.random.seed(42)
print("Setup complete.")

## The four evaluation metrics

PRAGMA evaluates downstream tasks with four complementary metrics (§3):

| Metric | What it measures | When to prefer it |
|---|---|---|
| **AUC-ROC** | Probability that a positive sample scores higher than a random negative | General-purpose; unaffected by class imbalance |
| **PR-AUC** | Area under the Precision-Recall curve | Imbalanced classes (few positives) — fraud detection |
| **F1** | Harmonic mean of precision and recall at a fixed threshold | When you need a single threshold-based score |
| **KS statistic** | Maximum separation between the positive and negative score CDF | Preferred in credit risk / banking regulation |

Source: `src/pragma_encoder/evaluation/metrics.py`

## Re-create the synthetic task

We recreate the 60-customer, two-class dataset from notebook 03 and run a short training loop so we have a trained model to evaluate.

In [ ]:
config     = PRAGMAConfig.pragma_s()
vocab_spec = VocabularySpec(
    special_tokens={"PAD": 0, "MASK": 1, "EVT": 2, "SEP": 3},
    key_start=4, key_size=config.key_vocab_size,
    value_start=4 + config.key_vocab_size, value_size=config.value_vocab_size,
    total_embedding_vocab_size=4 + config.key_vocab_size + config.value_vocab_size,
    field_key_ids={}, field_value_ranges={},
)

N_CUSTOMERS = 60; NE, NI, NA = 10, 8, 6
rng_g = torch.Generator(); rng_g.manual_seed(7)
labels  = torch.tensor([i % 2 for i in range(N_CUSTOMERS)], dtype=torch.long)
mid_val = vocab_spec.value_start + vocab_spec.value_size // 2

def _make_vals(label):
    lo = vocab_spec.value_start if label == 0 else mid_val
    hi = mid_val if label == 0 else vocab_spec.value_start + vocab_spec.value_size
    return torch.randint(lo, hi, (NE, NI), generator=rng_g)

xe_val_ids = torch.stack([_make_vals(int(l)) for l in labels])
xe_key_ids = torch.randint(vocab_spec.key_start, vocab_spec.key_start + vocab_spec.key_size,
                            (N_CUSTOMERS, NE, NI), generator=rng_g)
xa_key_ids = torch.randint(vocab_spec.key_start, vocab_spec.key_start + vocab_spec.key_size,
                            (N_CUSTOMERS, NA), generator=rng_g)
xa_val_ids = torch.randint(vocab_spec.value_start, vocab_spec.value_start + vocab_spec.value_size,
                            (N_CUSTOMERS, NA), generator=rng_g)
ta       = torch.rand(N_CUSTOMERS, NA, generator=rng_g) * 5.0
te       = torch.rand(N_CUSTOMERS, NE, generator=rng_g) * 80.0
calendar = torch.rand(N_CUSTOMERS, NE, 3, generator=rng_g)

train_idx = sorted(list(range(0, N_CUSTOMERS, 5)) + list(range(1, N_CUSTOMERS, 5)) +
                   list(range(2, N_CUSTOMERS, 5)) + list(range(3, N_CUSTOMERS, 5)))
test_idx  = list(range(4, N_CUSTOMERS, 5))
train_labels = labels[train_idx]; test_labels = labels[test_idx]

# ── Short pretraining loop ────────────────────────────────────────────────────
BATCH_SIZE = 12
torch.manual_seed(42)
model     = PRAGMA(config)
assembler = EmbeddingAssembler(vocab_spec, config)
masker    = MaskingStrategy(config)
optimizer = torch.optim.Adam(list(model.parameters()) + list(assembler.parameters()), lr=1e-4)

pretrain_losses = []
for step in range(50):
    chosen = torch.randperm(len(train_idx))[:BATCH_SIZE].tolist()
    idxs   = [train_idx[i] for i in chosen]
    optimizer.zero_grad()
    mv, _, mm = masker.forward(xe_val_ids[idxs], xe_key_ids[idxs])
    if not mm.any(): mm[0,0,0]=True; mv[0,0,0]=vocab_spec.special_tokens["MASK"]
    asm = assembler.forward(
        xa_key_ids=xa_key_ids[idxs], xa_val_ids=xa_val_ids[idxs], ta=ta[idxs],
        xe_key_ids=xe_key_ids[idxs], xe_val_ids=mv, te=te[idxs],
        calendar=calendar[idxs], targets=xe_val_ids[idxs], mlm_mask=mm,
    )
    out  = model.forward(xa=asm.xa, ta=asm.ta, xe=asm.xe, xt=asm.xt, te=asm.te, mask=asm.mlm_mask)
    loss = model.mlm_head.compute_loss(out["logits"], asm.targets[asm.mlm_mask])
    loss.backward(); optimizer.step()
    pretrain_losses.append(loss.item())

print(f"Pretraining complete. Loss: {pretrain_losses[0]:.4f} → {pretrain_losses[-1]:.4f}")

## Extract embeddings and fit probe

In [ ]:
model.eval()
all_emb = []
with torch.no_grad():
    for s in range(0, N_CUSTOMERS, BATCH_SIZE):
        e   = min(s + BATCH_SIZE, N_CUSTOMERS)
        asm = assembler.forward(
            xa_key_ids=xa_key_ids[s:e], xa_val_ids=xa_val_ids[s:e], ta=ta[s:e],
            xe_key_ids=xe_key_ids[s:e], xe_val_ids=xe_val_ids[s:e], te=te[s:e],
            calendar=calendar[s:e], targets=xe_val_ids[s:e],
            mlm_mask=torch.zeros(e-s, NE, NI, dtype=torch.bool),
        )
        o = model.forward(xa=asm.xa, ta=asm.ta, xe=asm.xe, xt=asm.xt, te=asm.te, mask=asm.mlm_mask)
        all_emb.append(o["zh"][:, 0, :].cpu())

embeddings = torch.cat(all_emb)   # (60, d_model)

probe = EmbeddingProbe(config)
probe.fit(embeddings[train_idx], train_labels, task="classification")

# Get probability scores for test set
test_emb  = embeddings[test_idx]
scores    = probe.predict(test_emb)   # raw decision scores
y_true    = test_labels.numpy()
y_score   = scores.numpy() if isinstance(scores, torch.Tensor) else np.array(scores)

# Normalise scores to [0, 1] if they aren't already (some probe implementations return raw logits)
if y_score.min() < 0 or y_score.max() > 1:
    y_score = (y_score - y_score.min()) / (y_score.max() - y_score.min() + 1e-9)

y_pred    = (y_score >= 0.5).astype(int)
print(f"Test set: {len(y_true)} samples, {y_true.sum()} positives")
print(f"Score range: [{y_score.min():.3f}, {y_score.max():.3f}]")

## Compute all four metrics

In [ ]:
auc    = compute_auc(y_true, y_score)      # AUC-ROC
pr_auc = compute_pr_auc(y_true, y_score)   # PR-AUC
f1     = compute_f1(y_true, y_pred)        # F1 at threshold 0.5
ks     = compute_ks(y_true, y_score)       # KS statistic

print("Evaluation metrics — linear probe on PRAGMA-S embeddings (50 pretraining steps)")
print(f"  AUC-ROC  : {auc:.3f}   (0.5 = chance, 1.0 = perfect)")
print(f"  PR-AUC   : {pr_auc:.3f}   (baseline = fraction of positives = 0.500)")
print(f"  F1 score : {f1:.3f}   (threshold=0.5; 0.0–1.0)")
print(f"  KS stat  : {ks:.3f}   (0.0 = no separation, 1.0 = perfect)")

## Visualisation 1 — All four metrics side by side

In [ ]:
metric_names  = ["AUC-ROC", "PR-AUC", "F1", "KS"]
metric_values = [auc, pr_auc, f1, ks]
# Baseline for each metric at random chance
baselines     = [0.5, 0.5, 0.5, 0.0]
colors        = ["#4C72B0", "#DD8452", "#55A868", "#8172B2"]

fig, axes = plt.subplots(1, 4, figsize=(13, 4), sharey=False)

for ax, name, val, base, col in zip(axes, metric_names, metric_values, baselines, colors):
    ax.bar([name], [val], color=col, width=0.55, edgecolor="white", linewidth=1.5)
    ax.axhline(base, color="#888", linewidth=1.2, linestyle="--",
               label=f"baseline ({base:.1f})")
    ax.text(0, val + 0.02, f"{val:.3f}", ha="center", fontsize=13, fontweight="bold")
    ax.set_ylim(0, 1.15)
    ax.set_title(name, fontsize=12)
    ax.legend(fontsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle("PRAGMA evaluation metrics — linear probe, 50-step synthetic run (§3)",
             fontsize=12)
plt.tight_layout()
plt.show()

**What you're looking at:** four separate bar charts, one per metric. The dashed line in each is the no-skill baseline. Any bar above its baseline means the probe has extracted real signal from the PRAGMA embeddings. On synthetic data after only 50 training steps the signal is small; on real IBM TabFormer data trained to convergence all four metrics show substantial improvement.

## Visualisation 2 — Probability score distributions

A good classifier separates the score distribution for positives and negatives. The KS statistic measures the maximum vertical gap between the two cumulative distributions.

In [ ]:
pos_scores = y_score[y_true == 1]
neg_scores = y_score[y_true == 0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Left: histograms
bins = np.linspace(0, 1, 16)
ax1.hist(neg_scores, bins=bins, alpha=0.6, color="#4C72B0", label="class 0 (negative)")
ax1.hist(pos_scores, bins=bins, alpha=0.6, color="#DD8452", label="class 1 (positive)")
ax1.set_xlabel("Probe score", fontsize=10)
ax1.set_ylabel("Count", fontsize=10)
ax1.set_title("Score distributions by class", fontsize=11)
ax1.legend(fontsize=9)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# Right: cumulative distributions (KS plot)
x_neg = np.sort(neg_scores); y_neg = np.arange(1, len(x_neg)+1) / len(x_neg)
x_pos = np.sort(pos_scores); y_pos = np.arange(1, len(x_pos)+1) / len(x_pos)
ax2.step(x_neg, y_neg, color="#4C72B0", linewidth=2, label="CDF class 0")
ax2.step(x_pos, y_pos, color="#DD8452", linewidth=2, label="CDF class 1")
# Mark KS point
ks_x = np.linspace(0, 1, 200)
cdf0 = np.array([np.mean(neg_scores <= xi) for xi in ks_x])
cdf1 = np.array([np.mean(pos_scores <= xi) for xi in ks_x])
ks_idx = np.argmax(np.abs(cdf0 - cdf1))
ax2.axvline(ks_x[ks_idx], color="#555", linewidth=1.2, linestyle=":",
            label=f"KS = {ks:.3f}")
ax2.set_xlabel("Probe score", fontsize=10)
ax2.set_ylabel("Cumulative fraction", fontsize=10)
ax2.set_title("KS statistic — CDF separation", fontsize=11)
ax2.legend(fontsize=9)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

plt.suptitle("Score distributions and KS statistic (§3)", fontsize=12)
plt.tight_layout()
plt.show()

**What you're looking at:** the left panel shows raw score distributions for positive and negative classes as overlapping histograms — better separation means less overlap. The right panel shows cumulative distribution functions; the dotted vertical line marks the point of maximum separation, which is the KS statistic.

## The full journey — summary table

Here is everything you built across the five notebooks:

In [ ]:
import textwrap

summary_rows = [
    {
        "Notebook": "01 — Tokenisation",
        "What you did": "Fit 4 tokenisers on 50 synthetic transactions",
        "Key result": "(key, value, time) shape; 4 field-type strategies",
        "Paper section": "§2.2",
    },
    {
        "Notebook": "02 — Pretraining",
        "What you did": "40-step MEM loop on random synthetic batch",
        "Key result": f"Loss {pretrain_losses[0]:.4f} → {pretrain_losses[-1]:.4f}; checkpoint verified",
        "Paper section": "§2.3, §2.3.5",
    },
    {
        "Notebook": "03 — Embedding probe",
        "What you did": "Fit EmbeddingProbe on frozen [USR] embeddings (50 steps)",
        "Key result": f"Probe AUC-ROC = {auc:.3f} (vs random baseline ~0.5)",
        "Paper section": "§3.1.1",
    },
    {
        "Notebook": "04 — LoRA fine-tuning",
        "What you did": "Applied LoRAAdapter, fine-tuned ~2.4% of parameters",
        "Key result": "r=8, alpha=8; log-scale param chart; BCE fine-tuning loop",
        "Paper section": "§3.1.2",
    },
    {
        "Notebook": "05 — Evaluation",
        "What you did": "Applied all 4 metrics to probe scores",
        "Key result": f"AUC={auc:.3f}  PR-AUC={pr_auc:.3f}  F1={f1:.3f}  KS={ks:.3f}",
        "Paper section": "§3",
    },
]

# Print as aligned table
col_widths = {k: max(len(k), max(len(str(r[k])) for r in summary_rows)) for k in summary_rows[0]}
header = "  ".join(k.ljust(col_widths[k]) for k in summary_rows[0])
sep    = "  ".join("-" * col_widths[k] for k in summary_rows[0])
print(header)
print(sep)
for row in summary_rows:
    print("  ".join(str(row[k]).ljust(col_widths[k]) for k in row))

## Visualisation 3 — All five notebooks in one chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Represent each notebook as a horizontal progress bar
notebooks  = ["01\nTokenisation", "02\nPretraining", "03\nProbe", "04\nLoRA", "05\nEvaluation"]
highlights = [
    "4 tokeniser\nstrategies",
    f"MEM loss\n{pretrain_losses[0]:.2f}→{pretrain_losses[-1]:.2f}",
    f"Probe AUC\n{auc:.3f}",
    "LoRA ~2.4%\ntrainable",
    f"KS={ks:.3f}\nF1={f1:.3f}",
]
notebook_colors = ["#4C72B0", "#DD8452", "#55A868", "#8172B2", "#C44E52"]

y_positions = list(range(len(notebooks) - 1, -1, -1))   # top to bottom

for y, nb, hl, col in zip(y_positions, notebooks, highlights, notebook_colors):
    ax.barh(y, 1.0, left=0, height=0.6, color=col, alpha=0.85, edgecolor="white")
    ax.text(-0.02, y, nb, ha="right", va="center", fontsize=9.5, fontweight="bold")
    ax.text(0.5,   y, hl, ha="center", va="center", fontsize=9, color="white", fontweight="bold")

ax.set_xlim(-0.45, 1.05)
ax.set_ylim(-0.7, len(notebooks) - 0.3)
ax.axis("off")
ax.set_title("The five-notebook PRAGMA journey — what you built", fontsize=12, pad=12)
plt.tight_layout()
plt.show()

## What just happened — section recap

Across five notebooks you have:

1. **Tokenised** financial transactions with four field-type-specific strategies (§2.2)
2. **Pretrained** PRAGMA-S with masked event modelling, saved and reloaded a checkpoint (§2.3, §2.3.5)
3. **Probed** frozen `[USR]` embeddings with a linear classifier (§3.1.1)
4. **Fine-tuned** with LoRA — 2.4% of parameters, zero frozen-weight mutation (§3.1.2)
5. **Evaluated** with AUC-ROC, PR-AUC, F1, and KS — the four metrics from §3

All of this ran on CPU with synthetic data in well under 10 minutes total.

## Closing — the next step

You've seen PRAGMA end-to-end on synthetic data. The architecture, training loop, probe, and LoRA adapter are identical to the production system. The only difference is scale and data.

**To train on real IBM TabFormer data:**
```bash
# From the repo root, using the training entrypoint:
python -m pragma_encoder.training.train \
    --dataset ibm-tabformer \
    --model-size S \
    --max-steps 500
```

Or, from inside the OpenShift AI Workbench, submit a KFP pipeline run:
```python
from tools.workbench import train_pragma
result = train_pragma(model_size="S", epochs=5)
result.show_pipeline()
```

**Going deeper:**

| Document | What it covers |
|---|---|
| `docs/training-guide.md` | Real training on IBM TabFormer — data prep, S3, cluster |
| `docs/workbench-journey.md` | The narrative walkthrough: dry_run → local → cluster |
| `docs/paper-to-code.md` | Every paper section mapped to its source file |
| `docs/testing-strategy.md` | Four-tier test model — how to trust the implementation |
| `docs/validated-architecture-summary.md` | What was validated end-to-end on the RHOAI cluster |
| `src/pragma_encoder/evaluation/metrics.py` | `compute_auc`, `compute_pr_auc`, `compute_f1`, `compute_ks` |

**Paper:** Ostroukhov et al. (2026), arXiv:2604.08649v1